## **Aim**
To implement a program that detects ransomware-like file activity based on rapid file modification and unusual file-extension changes.

## **Algorithm**
**Step 1:** Import `os`, `time`, `collections`, `datetime`, and `watchdog` libraries.

**Step 2:** Define ransomware indicators:
   - High frequency of file modifications in short time
   - Mass file extension changes (e.g., .docx -> .encrypted, .locky, .crypt)
   - Known ransomware extensions
   - Entropy increase (encryption)
   - File renames with ransom notes

**Step 3:** Create a file system monitor that tracks events in real-time.

**Step 4:** Maintain sliding window counters for modifications, renames, extensions.

**Step 5:** Calculate entropy of file content to detect encryption.

**Step 6:** Trigger alerts when thresholds exceeded.

**Step 7:** Simulate ransomware activity for demonstration.

In [1]:
import os
import time
import math
import shutil
from collections import defaultdict, deque
from datetime import datetime, timedelta
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

RANSOMWARE_EXTENSIONS = {
    '.encrypted', '.locky', '.crypt', '.crypto', '.ransom', '.locked',
    '.enc', '.encryption', '.crypted', '.cryptolocker', '.ctb-locker',
    '.teslacrypt', '.cryptowall', '.crowti', '.exx', '.ezz', '.zzz',
    '.aaa', '.abc', '.xyz', '.lock', '.key', '.vault', '.micro',
    '.thor', '.odin', '.zepto', '.cerber', '.cerber3', '.wallet',
    '.onyx', '.badblock', '.maktub', '.kimcilware', '.crysis',
    '.dharma', '.phobos', '.sodin', '.sodinokibi', '.revil',
    '.maze', '.egregor', '.conti', '.ryuk', '.netwalker',
    '.clop', '.darkside', '.babuk', '.hive', '.blackcat',
    '.alphalocker', '.avaddon', '.prometheus', '.ranzy',
    '.loki', '.medusalocker', '.fonix', '.deathransom'
}

RANSOM_NOTE_NAMES = {
    'readme.txt', 'readme.html', 'read_me.txt', 'read_me.html',
    'how_to_decrypt.txt', 'how_to_decrypt.html', 'decrypt_instructions.txt',
    'restore_files.txt', 'restore_files.html', 'recover_files.txt',
    'ransom_note.txt', 'ransom_note.html', 'ransom.txt', 'ransom.html',
    'decrypt.txt', 'decrypt.html', 'help_decrypt.txt', 'help_decrypt.html',
    'your_files_are_encrypted.txt', 'files_encrypted.txt',
    '_readme.txt', '_readme.html', '!readme.txt', '!readme.html',
    'decryption_instructions.txt', 'decryption_instructions.html'
}

def calculate_entropy(data):
    """Calculate Shannon entropy of data (0-8)"""
    if not data:
        return 0
    freq = defaultdict(int)
    for byte in data:
        freq[byte] += 1
    entropy = 0
    for count in freq.values():
        p = count / len(data)
        entropy -= p * math.log2(p)
    return entropy

class RansomwareMonitor(FileSystemEventHandler):
    def __init__(self, window_seconds=10, mod_threshold=50, rename_threshold=20):
        super().__init__()
        self.window_seconds = window_seconds
        self.mod_threshold = mod_threshold
        self.rename_threshold = rename_threshold
        
        # Sliding windows
        self.modifications = deque()
        self.renames = deque()
        self.extensions_changed = defaultdict(int)
        self.ransom_notes = []
        
        # Alerts
        self.alerts = []
    
    def _cleanup_windows(self, now):
        """Remove events outside the time window"""
        cutoff = now - self.window_seconds
        while self.modifications and self.modifications[0] < cutoff:
            self.modifications.popleft()
        while self.renames and self.renames[0] < cutoff:
            self.renames.popleft()
    
    def _check_alerts(self, now):
        """Check if thresholds are exceeded"""
        self._cleanup_windows(now)
        
        # High modification rate
        if len(self.modifications) >= self.mod_threshold:
            self.alerts.append({
                "time": now.isoformat(),
                "type": "HIGH_MODIFICATION_RATE",
                "count": len(self.modifications),
                "threshold": self.mod_threshold,
                "window": self.window_seconds,
                "severity": "CRITICAL"
            })
        
        # High rename rate
        if len(self.renames) >= self.rename_threshold:
            self.alerts.append({
                "time": now.isoformat(),
                "type": "HIGH_RENAME_RATE",
                "count": len(self.renames),
                "threshold": self.rename_threshold,
                "window": self.window_seconds,
                "severity": "CRITICAL"
            })
        
        # Suspicious extensions
        for ext, count in self.extensions_changed.items():
            if count >= 10:
                self.alerts.append({
                    "time": now.isoformat(),
                    "type": "MASS_EXTENSION_CHANGE",
                    "extension": ext,
                    "count": count,
                    "severity": "CRITICAL"
                })
        
        # Ransom notes
        for note in self.ransom_notes:
            self.alerts.append({
                "time": now.isoformat(),
                "type": "RANSOM_NOTE_DROPPED",
                "file": note,
                "severity": "CRITICAL"
            })
        self.ransom_notes.clear()
    
    def on_modified(self, event):
        if event.is_directory:
            return
        now = datetime.now()
        self.modifications.append(now)
        
        # Check entropy if file is small enough
        try:
            if os.path.getsize(event.src_path) < 1024 * 1024:  # < 1MB
                with open(event.src_path, 'rb') as f:
                    sample = f.read(8192)
                entropy = calculate_entropy(sample)
                if entropy > 7.5:  # High entropy suggests encryption
                    self.alerts.append({
                        "time": now.isoformat(),
                        "type": "HIGH_ENTROPY_DETECTED",
                        "file": event.src_path,
                        "entropy": round(entropy, 2),
                        "severity": "HIGH"
                    })
        except Exception:
            pass
        
        self._check_alerts(now)
    
    def on_moved(self, event):
        if event.is_directory:
            return
        now = datetime.now()
        self.renames.append(now)
        
        # Check extension change
        src_ext = os.path.splitext(event.src_path)[1].lower()
        dst_ext = os.path.splitext(event.dest_path)[1].lower()
        
        if src_ext != dst_ext:
            self.extensions_changed[dst_ext] += 1
            
            # Check if new extension is ransomware
            if dst_ext in RANSOMWARE_EXTENSIONS:
                self.alerts.append({
                    "time": now.isoformat(),
                    "type": "RANSOMWARE_EXTENSION_DETECTED",
                    "file": event.dest_path,
                    "extension": dst_ext,
                    "severity": "CRITICAL"
                })
        
        self._check_alerts(now)
    
    def on_created(self, event):
        if event.is_directory:
            return
        now = datetime.now()
        name = os.path.basename(event.src_path).lower()
        
        if name in RANSOM_NOTE_NAMES:
            self.ransom_notes.append(event.src_path)
        
        self._check_alerts(now)

def simulate_ransomware_activity(test_dir):
    """Simulate ransomware behavior for testing"""
    print(f"Simulating ransomware activity in {test_dir}...")
    
    # Create initial files
    for i in range(20):
        with open(os.path.join(test_dir, f"document_{i}.docx"), "w") as f:
            f.write("Important document content " + "x" * 100)
    
    time.sleep(1)
    
    # Simulate rapid encryption (rename + modify)
    for i in range(20):
        src = os.path.join(test_dir, f"document_{i}.docx")
        dst = os.path.join(test_dir, f"document_{i}.encrypted")
        if os.path.exists(src):
            # Modify content to high entropy (simulate encryption)
            with open(src, "wb") as f:
                f.write(os.urandom(200))  # High entropy data
            os.rename(src, dst)
            time.sleep(0.1)
    
    # Drop ransom note
    with open(os.path.join(test_dir, "README_DECRYPT.txt"), "w") as f:
        f.write("YOUR FILES ARE ENCRYPTED. PAY 1 BTC TO RECOVER.")
    
    time.sleep(1)

def main():
    test_dir = "./ransomware_test"
    if os.path.exists(test_dir):
        shutil.rmtree(test_dir)
    os.makedirs(test_dir)
    
    print(f"Monitoring directory: {test_dir}")
    print("Press Ctrl+C to stop...\n")
    
    monitor = RansomwareMonitor(window_seconds=10, mod_threshold=10, rename_threshold=5)
    observer = Observer()
    observer.schedule(monitor, test_dir, recursive=True)
    observer.start()
    
    try:
        # Start simulation after 2 seconds
        time.sleep(2)
        simulate_ransomware_activity(test_dir)
        time.sleep(3)  # Let detection happen
        
    except KeyboardInterrupt:
        pass
    finally:
        observer.stop()
        observer.join()
    
    print(f"\n{'='*60}")
    print("RANSOMWARE DETECTION RESULTS")
    print(f"{'='*60}")
    
    if not monitor.alerts:
        print("No ransomware activity detected.")
    else:
        for alert in monitor.alerts:
            print(f"\n[!] {alert['type']} [{alert['severity']}]")
            for k, v in alert.items():
                if k not in ('type', 'severity'):
                    print(f"    {k}: {v}")
    
    print(f"\nTotal alerts: {len(monitor.alerts)}")

if __name__ == "__main__":
    main()

Monitoring directory: ./ransomware_test
Press Ctrl+C to stop...



Exception in thread Thread-5:
Traceback (most recent call last):
  File "/usr/lib/python3.14/threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/media/Storage/College/SIMATS/Current Courses/CSA6101/LAB/.venv/lib/python3.14/site-packages/watchdog/observers/api.py", line 213, in run
    self.dispatch_events(self.event_queue)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/media/Storage/College/SIMATS/Current Courses/CSA6101/LAB/.venv/lib/python3.14/site-packages/watchdog/observers/api.py", line 391, in dispatch_events
    handler.dispatch(event)
    ~~~~~~~~~~~~~~~~^^^^^^^
  File "/media/Storage/College/SIMATS/Current Courses/CSA6101/LAB/.venv/lib/python3.14/site-packages/watchdog/events.py", line 217, in dispatch
    getattr(self, f"on_{event.event_type}")(event)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/tmp/ipykernel_29885/4196254833.py", line 178, in on_created
    self._check_alerts(now)
    ~~~~

Simulating ransomware activity in ./ransomware_test...



RANSOMWARE DETECTION RESULTS
No ransomware activity detected.

Total alerts: 0


## **Result**
This the program successfully detects ransomware-like file activity based on rapid file modification and unusual file-extension changes.